# 🌾 AgriScope – Complete ML Model Training Notebook
**Smart Agriculture Decision Support System – Gujarat, India**

This notebook trains **9 machine learning regression models** to predict crop yield (kg/ha)
and documents all metrics + visualizations for the instructor to evaluate.

**Target Variable:** `yield` (kg/hectare)  
**Best Model:** ExtraTrees Regressor (~67% accuracy)  
**Data:** Gujarat crop data 2016–2024, 32 districts, 3 seasons

---

## 📦 1. Setup & Imports

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import joblib
import json
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import (
    RandomForestRegressor, GradientBoostingRegressor,
    ExtraTreesRegressor, AdaBoostRegressor
)
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score
)

try:
    from xgboost import XGBRegressor
    USE_XGB = True
    print('✅ XGBoost available')
except ImportError:
    USE_XGB = False
    print('⚠️  XGBoost not installed — pip install xgboost')

try:
    from lightgbm import LGBMRegressor
    USE_LGB = True
    print('✅ LightGBM available')
except ImportError:
    USE_LGB = False
    print('⚠️  LightGBM not installed — pip install lightgbm')

from utils.data_cleaning import clean_data

# Paths
RAW_PATH     = '../data/final_data.csv'
CLEANED_PATH = '../cleaned_data/cleaned_data.csv'
MODEL_DIR    = '../models'
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs('../cleaned_data', exist_ok=True)

# Matplotlib theme
plt.style.use('seaborn-v0_8-darkgrid')
PALETTE = ['#27ae60','#2ecc71','#f39c12','#3498db','#9b59b6',
           '#e74c3c','#1abc9c','#e67e22','#c0392b','#2980b9']
print('✅ All imports successful!')

## 🔍 2. Exploratory Data Analysis (EDA)

In [ ]:
# Load raw data
df_raw = pd.read_csv(RAW_PATH)
print(f'Raw data shape: {df_raw.shape}')
print(f'\nColumns: {list(df_raw.columns)}')
df_raw.head()

In [ ]:
print('\n--- Data Info ---')
df_raw.info()
print('\n--- Missing Values ---')
print(df_raw.isnull().sum())
print('\n--- Basic Statistics ---')
df_raw.describe()

In [ ]:
# EDA: Distribution of key columns
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Raw Data – Feature Distributions Before Cleaning', fontsize=14, y=1.02)

# Normalize column names for EDA
df_eda = df_raw.copy()
df_eda.columns = df_eda.columns.str.strip().str.lower().str.replace(' ', '_', regex=False)

numeric_cols = df_eda.select_dtypes(include=[np.number]).columns.tolist()[:6]
for i, col in enumerate(numeric_cols):
    ax = axes[i//3][i%3]
    df_eda[col].dropna().hist(ax=ax, bins=40, color=PALETTE[i], edgecolor='white', alpha=0.85)
    ax.set_title(col.replace('_', ' ').title())
    ax.set_ylabel('Frequency')

plt.tight_layout()
plt.savefig('../models/eda_distributions.png', dpi=120, bbox_inches='tight')
plt.show()
print('📊 Distribution plot saved to models/eda_distributions.png')

In [ ]:
# Season and district distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

season_col = next((c for c in df_eda.columns if 'season' in c.lower()), None)
dist_col   = next((c for c in df_eda.columns if 'district' in c.lower()), None)

if season_col:
    season_counts = df_eda[season_col].value_counts()
    axes[0].pie(season_counts.values, labels=season_counts.index,
                autopct='%1.1f%%', colors=PALETTE[:3], startangle=90)
    axes[0].set_title('Season Distribution')

if dist_col:
    dist_counts = df_eda[dist_col].value_counts().head(15)
    axes[1].barh(dist_counts.index[::-1], dist_counts.values[::-1],
                 color=PALETTE[1], alpha=0.85)
    axes[1].set_title('Top 15 Districts by Record Count')
    axes[1].set_xlabel('Number of Records')

plt.tight_layout()
plt.savefig('../models/eda_categories.png', dpi=120, bbox_inches='tight')
plt.show()

## 🧹 3. Data Cleaning Pipeline

In [ ]:
print('Running data cleaning pipeline...')
df = clean_data(RAW_PATH, CLEANED_PATH)
print(f'\n✅ Cleaned data shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

In [ ]:
# Reload cleaned data
df = pd.read_csv(CLEANED_PATH)
print(f'Cleaned dataset: {df.shape[0]} rows × {df.shape[1]} columns')
print(f'Missing values after cleaning: {df.isnull().sum().sum()}')
print('\nData types:')
print(df.dtypes)

## 🔧 4. Feature Engineering & Correlation Analysis

In [ ]:
# Encode crop_type if not already done
if 'crop_type' in df.columns and 'crop_type_encoded' not in df.columns:
    le_crop = LabelEncoder()
    df['crop_type_encoded'] = le_crop.fit_transform(df['crop_type'].astype(str))
    print(f'Encoded {df["crop_type"].nunique()} crop types')
else:
    le_crop = None
    print('crop_type_encoded already present')

# Define features
FEATURES = [
    'district_encoded', 'season_encoded', 'crop_type_encoded',
    'total_rainfall', 'rainy_days',
    'average_tmax', 'average_tmin', 'average_humidity'
]
FEATURES = [f for f in FEATURES if f in df.columns]
TARGET   = 'yield'

print(f'\nFeatures used ({len(FEATURES)}): {FEATURES}')
df_model = df[FEATURES + [TARGET]].dropna()
df_model = df_model[df_model[TARGET] > 0]
print(f'Final dataset for modelling: {df_model.shape[0]} samples')

In [ ]:
# Correlation heatmap
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Correlation matrix
corr = df_model.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, ax=axes[0], linewidths=0.5)
axes[0].set_title('Feature Correlation Heatmap', fontsize=13, pad=12)

# Yield distribution (raw vs log)
y_raw = df_model[TARGET]
y_log = np.log1p(y_raw)
axes[1].hist(y_raw, bins=50, color=PALETTE[0], alpha=0.6, label=f'Raw  (skew={y_raw.skew():.2f})')
ax2 = axes[1].twinx()
ax2.hist(y_log, bins=50, color=PALETTE[2], alpha=0.6, label=f'Log  (skew={y_log.skew():.2f})')
axes[1].set_title('Yield Distribution: Raw vs Log-Transformed', fontsize=13)
axes[1].set_xlabel('Yield')
axes[1].set_ylabel('Count (raw)', color=PALETTE[0])
ax2.set_ylabel('Count (log)', color=PALETTE[2])
lines1, labels1 = axes[1].get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
axes[1].legend(lines1 + lines2, labels1 + labels2, loc='upper right')

plt.tight_layout()
plt.savefig('../models/feature_correlation.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Yield skewness: raw={y_raw.skew():.3f}  →  log={y_log.skew():.3f}')

## 🔀 5. Train / Test Split & Scaling

In [ ]:
X = df_model[FEATURES]
y = df_model[TARGET]

# Apply log-transform to reduce skewness
y_log = np.log1p(y)

# 80/20 split
X_train, X_test, y_train_log, y_test_log = train_test_split(
    X, y_log, test_size=0.2, random_state=42
)
y_train_orig = np.expm1(y_train_log)
y_test_orig  = np.expm1(y_test_log)

# Scale features
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Training samples : {X_train.shape[0]}')
print(f'Test samples     : {X_test.shape[0]}')
print(f'Features         : {len(FEATURES)}')
print(f'\nYield stats:')
print(f'  Mean  = {y.mean():.1f} kg/ha')
print(f'  Std   = {y.std():.1f} kg/ha')
print(f'  Min   = {y.min():.1f} kg/ha')
print(f'  Max   = {y.max():.1f} kg/ha')

## 🤖 6. Train All ML Models

In [ ]:
# Define all models
models_to_train = {
    'RandomForest':     RandomForestRegressor(n_estimators=300, max_depth=12,
                                              min_samples_split=4, random_state=42, n_jobs=-1),
    'ExtraTrees':       ExtraTreesRegressor(n_estimators=300, max_depth=12,
                                            random_state=42, n_jobs=-1),
    'GradientBoosting': GradientBoostingRegressor(n_estimators=300, learning_rate=0.08,
                                                  max_depth=5, random_state=42),
    'DecisionTree':     DecisionTreeRegressor(max_depth=12, min_samples_split=4,
                                              random_state=42),
    'Ridge':            Ridge(alpha=1.0),
    'ElasticNet':       ElasticNet(alpha=0.5, l1_ratio=0.5, max_iter=5000),
    'KNeighbors':       KNeighborsRegressor(n_neighbors=7, weights='distance'),
}
if USE_XGB:
    models_to_train['XGBoost'] = XGBRegressor(
        n_estimators=300, learning_rate=0.08, max_depth=6,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, eval_metric='rmse', verbosity=0
    )
if USE_LGB:
    models_to_train['LightGBM'] = LGBMRegressor(
        n_estimators=300, learning_rate=0.08, max_depth=6,
        random_state=42, verbose=-1
    )

print(f'Training {len(models_to_train)} models...\n')
print(f'{"Model":<22} {"R²":>8} {"Accuracy%":>11} {"MAE":>10} {"RMSE":>10}')
print('-' * 65)

results  = {}
metrics  = {}
fitted   = {}

for name, mdl in models_to_train.items():
    mdl.fit(X_train_sc, y_train_log)
    preds_log  = mdl.predict(X_test_sc)
    preds_orig = np.expm1(preds_log)

    mae  = mean_absolute_error(y_test_orig, preds_orig)
    rmse = np.sqrt(mean_squared_error(y_test_orig, preds_orig))
    r2   = r2_score(y_test_orig, preds_orig)
    acc  = max(0.0, min(100.0, r2 * 100))

    results[name] = r2
    fitted[name]  = mdl
    metrics[name] = {'MAE': round(float(mae),2), 'RMSE': round(float(rmse),2),
                     'R2': round(float(r2),4), 'Accuracy': round(acc,2)}
    print(f'{name:<22} {r2:>8.4f} {acc:>10.2f}% {mae:>10.1f} {rmse:>10.1f}')

print('\n✅ Training complete!')

## 📊 7. Model Comparison Charts

In [ ]:
# Sort by accuracy
ranked = sorted(results.items(), key=lambda x: x[1], reverse=True)
best_name  = ranked[0][0]
best_model = fitted[best_name]
top5 = [n for n, _ in ranked[:5]]

names_sorted = [n for n, _ in ranked]
accs_sorted  = [metrics[n]['Accuracy'] for n in names_sorted]
r2_sorted    = [metrics[n]['R2'] for n in names_sorted]
mae_sorted   = [metrics[n]['MAE'] for n in names_sorted]
rmse_sorted  = [metrics[n]['RMSE'] for n in names_sorted]
colors_bar   = [PALETTE[0] if n == best_name else PALETTE[3] for n in names_sorted]

print(f'🏆 Best model: {best_name}  (Accuracy={metrics[best_name]["Accuracy"]:.2f}%, R²={metrics[best_name]["R2"]:.4f})')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('All ML Models – Performance Comparison', fontsize=15, y=1.03)

# Accuracy
bars = axes[0].barh(names_sorted[::-1], accs_sorted[::-1], color=colors_bar[::-1], edgecolor='white', height=0.6)
axes[0].set_xlabel('Accuracy (%)')
axes[0].set_title('Model Accuracy (R²×100)')
axes[0].axvline(50, color='grey', linestyle='--', linewidth=1, label='50% line')
for bar, val in zip(bars, accs_sorted[::-1]):
    axes[0].text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                 f'{val:.1f}%', va='center', fontsize=9, fontweight='bold')
axes[0].legend()

# R² Scores
bar_colors2 = [PALETTE[0] if n == best_name else PALETTE[3] for n in names_sorted]
axes[1].bar(names_sorted, r2_sorted, color=bar_colors2, edgecolor='white')
axes[1].axhline(0, color='red', linestyle='--', linewidth=1.5, label='Baseline R²=0')
axes[1].set_ylabel('R² Score')
axes[1].set_title('R² Score (higher is better)')
axes[1].tick_params(axis='x', rotation=30)
for i, val in enumerate(r2_sorted):
    axes[1].text(i, val + 0.01 if val >= 0 else val - 0.03,
                 f'{val:.3f}', ha='center', fontsize=8)
axes[1].legend()

# MAE vs RMSE
x = np.arange(len(names_sorted)); w = 0.38
axes[2].bar(x - w/2, mae_sorted,  w, label='MAE',  color=PALETTE[2], edgecolor='white')
axes[2].bar(x + w/2, rmse_sorted, w, label='RMSE', color=PALETTE[4], edgecolor='white')
axes[2].set_xticks(x)
axes[2].set_xticklabels(names_sorted, rotation=30, ha='right', fontsize=8)
axes[2].set_ylabel('Error (kg/ha)')
axes[2].set_title('MAE & RMSE (lower is better)')
axes[2].legend()

plt.tight_layout()
plt.savefig('../models/model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print('📊 Model comparison chart saved.')

## 🏆 8. Best Model Deep-Dive Analysis

In [ ]:
# Feature Importance (best model)
print(f'Analysing best model: {best_name}')

preds_best_log  = best_model.predict(X_test_sc)
preds_best_orig = np.expm1(preds_best_log)
residuals = y_test_orig.values - preds_best_orig

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(f'{best_name} – Deep-Dive Analysis', fontsize=15, y=1.01)

# --- Feature Importance ---
if hasattr(best_model, 'feature_importances_'):
    fi = pd.Series(best_model.feature_importances_, index=FEATURES).sort_values()
    colors_fi = [PALETTE[0] if v == fi.max() else PALETTE[1] for v in fi.values]
    axes[0,0].barh(fi.index, fi.values, color=colors_fi, edgecolor='white')
    axes[0,0].set_title('Feature Importance')
    axes[0,0].set_xlabel('Importance Score')
    for i, v in enumerate(fi.values):
        axes[0,0].text(v + 0.001, i, f'{v:.3f}', va='center', fontsize=9)
else:
    axes[0,0].text(0.5, 0.5, 'Feature importance\nnot available', ha='center', va='center')
    axes[0,0].set_axis_off()

# --- Actual vs Predicted ---
axes[0,1].scatter(y_test_orig, preds_best_orig, alpha=0.45, s=18,
                  color=PALETTE[1], edgecolors='none')
lims = [min(y_test_orig.min(), preds_best_orig.min()) * 0.95,
        max(y_test_orig.max(), preds_best_orig.max()) * 1.05]
axes[0,1].plot(lims, lims, 'r--', lw=2, label='Perfect Prediction')
axes[0,1].set_xlabel('Actual Yield (kg/ha)')
axes[0,1].set_ylabel('Predicted Yield (kg/ha)')
axes[0,1].set_title(f'Actual vs Predicted  (R²={metrics[best_name]["R2"]:.4f})')
axes[0,1].legend()

# --- Residual Distribution ---
axes[1,0].hist(residuals, bins=45, color=PALETTE[3], edgecolor='white', alpha=0.85)
axes[1,0].axvline(0, color='red', linestyle='--', lw=2, label='Zero error')
axes[1,0].axvline(residuals.mean(), color=PALETTE[0], linestyle='--', lw=2,
                  label=f'Mean error = {residuals.mean():.1f}')
axes[1,0].set_xlabel('Residual (Actual – Predicted) kg/ha')
axes[1,0].set_ylabel('Count')
axes[1,0].set_title('Residual Distribution')
axes[1,0].legend()

# --- Residuals vs Predicted ---
axes[1,1].scatter(preds_best_orig, residuals, alpha=0.4, s=14,
                  color=PALETTE[2], edgecolors='none')
axes[1,1].axhline(0, color='red', linestyle='--', lw=2)
axes[1,1].set_xlabel('Predicted Yield (kg/ha)')
axes[1,1].set_ylabel('Residual (kg/ha)')
axes[1,1].set_title('Residuals vs Predicted Values')

plt.tight_layout()
plt.savefig('../models/best_model_analysis.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'📊 {best_name} analysis charts saved.')

## 📈 9. Learning Curves & Cross-Validation

In [ ]:
# Learning curve for best model
print(f'Computing learning curve for {best_name} (this may take ~60 s)...')

train_sizes, train_scores, val_scores = learning_curve(
    best_model, X_train_sc, y_train_log, cv=5,
    train_sizes=np.linspace(0.1, 1.0, 10),
    scoring='r2', n_jobs=-1, random_state=42
)

fig, ax = plt.subplots(figsize=(10, 5))
train_mean = train_scores.mean(axis=1)
train_std  = train_scores.std(axis=1)
val_mean   = val_scores.mean(axis=1)
val_std    = val_scores.std(axis=1)

ax.plot(train_sizes, train_mean, 'o-', color=PALETTE[0], lw=2.5, label='Training Score')
ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std,
                alpha=0.18, color=PALETTE[0])
ax.plot(train_sizes, val_mean, 's-', color=PALETTE[2], lw=2.5, label='Cross-Validation Score')
ax.fill_between(train_sizes, val_mean - val_std, val_mean + val_std,
                alpha=0.18, color=PALETTE[2])

ax.set_xlabel('Training Set Size')
ax.set_ylabel('R² Score')
ax.set_title(f'{best_name} – Learning Curve')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig('../models/learning_curve.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# 5-Fold Cross-Validation on Top 3 models
print('5-Fold Cross-Validation (top 3 models)...')
top3 = [n for n, _ in ranked[:3]]
cv_results = {}

for name in top3:
    scores = cross_val_score(fitted[name], X_train_sc, y_train_log,
                             cv=5, scoring='r2', n_jobs=-1)
    cv_results[name] = scores
    print(f'{name:<22}  CV R²: {scores.mean():.4f} ± {scores.std():.4f}  |  scores: {np.round(scores,4)}')

# Box plot
fig, ax = plt.subplots(figsize=(9, 5))
bplot = ax.boxplot([cv_results[n] for n in top3], labels=top3, patch_artist=True,
                   medianprops=dict(color='red', linewidth=2.5))
for patch, color in zip(bplot['boxes'], PALETTE[:3]):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)
ax.set_ylabel('R² (Cross-Validation)')
ax.set_title('5-Fold CV Score Distribution – Top 3 Models')
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.savefig('../models/cross_validation.png', dpi=120, bbox_inches='tight')
plt.show()

## 💾 10. Save All Models & Artifacts

In [ ]:
print('Saving models and artifacts...')

# Save best model
joblib.dump(best_model, os.path.join(MODEL_DIR, 'model.pkl'))
print(f'✅ Best model saved   : models/model.pkl')

# Save scaler
joblib.dump(scaler, os.path.join(MODEL_DIR, 'scaler.pkl'))
print(f'✅ Scaler saved       : models/scaler.pkl')

# Save every model individually
for name, mdl in fitted.items():
    safe = name.lower().replace(' ', '_')
    joblib.dump(mdl, os.path.join(MODEL_DIR, f'{safe}_model.pkl'))
print(f'✅ All individual models saved to models/')

# Save encoders
encoders = {}
for col in ['district', 'season', 'crop_type']:
    if col in df.columns:
        le = LabelEncoder()
        le.fit(df[col].astype(str))
        encoders[col] = le
joblib.dump(encoders, os.path.join(MODEL_DIR, 'encoders.pkl'))
print(f'✅ Encoders saved     : models/encoders.pkl')

# Save transform info
joblib.dump({'log_transform': True, 'le_crop': le_crop},
            os.path.join(MODEL_DIR, 'transform_info.pkl'))
print(f'✅ Transform info saved')

# Save metrics JSON
metrics_out = {
    'best_model':    best_name,
    'top5':          top5,
    'models':        metrics,
    'features':      FEATURES,
    'log_transform': True,
    'dataset_size':  int(X.shape[0]),
    'test_size':     int(X_test.shape[0]),
}
with open(os.path.join(MODEL_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics_out, f, indent=2)
print(f'✅ Metrics JSON saved : models/metrics.json')

## 📋 11. Final Summary Table

In [ ]:
summary_rows = []
for i, (name, _) in enumerate(ranked, 1):
    m = metrics[name]
    crown  = ' 🏆' if name == best_name else ''
    medal  = ['🥇','🥈','🥉','4️⃣','5️⃣','6️⃣','7️⃣','8️⃣','9️⃣'][i-1] if i <= 9 else str(i)
    summary_rows.append({
        'Rank'        : f'{medal} #{i}',
        'Model'       : name + crown,
        'Accuracy %'  : f"{m['Accuracy']:.2f}%",
        'R² Score'    : f"{m['R2']:.4f}",
        'MAE (kg/ha)' : f"{m['MAE']:,.1f}",
        'RMSE (kg/ha)': f"{m['RMSE']:,.1f}",
    })

summary_df = pd.DataFrame(summary_rows).set_index('Rank')
print('\n' + '='*70)
print(f'  AgriScope – Final Model Summary')
print('='*70)
print(summary_df.to_string())
print('='*70)
print(f'\n🏆 Winner: {best_name}')
print(f'   Accuracy  : {metrics[best_name]["Accuracy"]:.2f}%')
print(f'   R² Score  : {metrics[best_name]["R2"]:.4f}')
print(f'   MAE       : {metrics[best_name]["MAE"]:,.1f} kg/ha')
print(f'   RMSE      : {metrics[best_name]["RMSE"]:,.1f} kg/ha')
print(f'\n📁 All models and artifacts saved to: {os.path.abspath(MODEL_DIR)}')
print(f'\n🚀 To launch the dashboard:')
print(f'   cd AgriScope')
print(f'   streamlit run app/app.py')